# CCA-F 考试复习 Notebook — Context Management

本 notebook 聚焦 Domain 5：上下文管理。每个 Task 都保留固定结构：概念解释、可运行 Python 示例、考点总结与 2 道官方风格模拟题。所有代码均为本地模拟，不需要 API key。


## 环境初始化

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Optional

MODEL = "claude-haiku-4-5-20251001"
print("环境就绪：本 notebook 使用本地模拟，代码结构贴近 Anthropic Messages API / Claude Code 工作流。")

## D5.1 — 管理长上下文输入与 Lost in the Middle 效应

### 📖 概念解释

长上下文并不等于稳定理解全部内容。模型常见 **primacy / recency bias**：开头和结尾的信息更容易被利用，中间证据更容易被忽略，即 lost in the middle。CCA-F 考查的不是“把更多 token 塞进去”，而是如何设计可恢复、可审计的上下文布局。

生产做法通常包括：把任务目标、关键事实和判定标准放在开头；用清晰标题分段；把最终约束、输出格式和禁止事项放在末尾重申；对工具结果做结构化裁剪，只保留当前决策需要的字段。冗长日志、完整 HTML、未筛选搜索结果和重复历史消息会增加成本，也会稀释关键证据。


In [ ]:
# Task 5.1：长上下文、lost in the middle、关键位置与工具输出裁剪
# 本示例用一个简单评分函数模拟：开头/结尾证据更容易被检索，中间证据权重较低。

verbose_tool_result = {
    **{f"debug_field_{i}": f"noise_{i}" for i in range(18)},
    "order_id": "ORD-9",
    "refund_status": "eligible",
    "policy_basis": "returned_within_30_days",
    "amount_usd": 88.10,
}

def trim_refund_tool_result(raw: dict) -> dict:
    """只保留自动退款决策需要的字段。"""
    keep = ["order_id", "refund_status", "policy_basis", "amount_usd"]
    return {key: raw[key] for key in keep}

def position_score(sections: list[tuple[str, str]], keyword: str) -> float:
    """演示 lost-in-the-middle 风险：越靠近开头或结尾，检索权重越高。"""
    n = len(sections)
    score = 0.0
    for index, (_, text) in enumerate(sections):
        if keyword in text:
            distance_to_edge = min(index, n - 1 - index)
            score += 1 / (1 + distance_to_edge)
    return round(score, 3)

bad_context = [
    ("chat_history", "客户询问退款。"),
    ("raw_tool_output", json.dumps(verbose_tool_result, ensure_ascii=False)),
    ("irrelevant_logs", "大量调试日志..." * 20),
    ("critical_fact", "ORD-9 refund_status=eligible policy_basis=returned_within_30_days"),
    ("closing", "请回答。"),
]

good_context = [
    ("decision_summary", "任务：判断 ORD-9 是否可自动退款。关键事实：refund_status=eligible。"),
    ("trimmed_tool_result", json.dumps(trim_refund_tool_result(verbose_tool_result), ensure_ascii=False)),
    ("background", "只保留与退款判断有关的简短背景。"),
    ("final_constraints", "只依据 order_id、refund_status、policy_basis、amount_usd 回答。"),
]

print("bad_context_score:", position_score(bad_context, "refund_status=eligible"))
print("good_context_score:", position_score(good_context, "refund_status=eligible"))
print(json.dumps(dict(good_context), ensure_ascii=False, indent=2))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 将任务目标、关键事实、判定标准放在上下文开头。
- 在末尾重申输出格式、约束和禁止事项，利用 recency bias。
- 对工具输出做字段级裁剪或摘要，不把无关日志、HTML、搜索结果整包塞入 prompt。
- 用标题和结构化 JSON 帮助模型区分事实、背景、约束和待办。

**✗ 常见陷阱**

- 误以为“长上下文窗口”消除了 lost in the middle。
- 把唯一关键证据埋在中间，然后责怪模型没有遵循。
- 为了“完整性”保留所有工具输出，反而增加成本和噪声。
- 只裁剪自然语言历史，不裁剪工具返回的冗长字段。


### 🧪 模拟题

**Q1.** 一个 agent 需要根据订单状态决定是否自动退款。工具返回 80 个字段，其中只有 `order_id`、`refund_status`、`policy_basis`、`amount_usd` 与决策有关。最合适的上下文策略是什么？

A) 将关键摘要放在开头，裁剪工具结果，并在末尾重申判定字段  
B) 保留完整工具结果，避免遗漏任何信息  
C) 把关键字段放在中间，保持原始返回顺序  
D) 只在 system prompt 中写“请注意重要信息”

> **答案：A**  
> **解析：** CCA-F 强调通过布局和裁剪提高可靠性。完整工具输出会稀释关键证据，关键事实应前置并在末尾通过约束强化。

**Q2.** 关于 long context 和 lost in the middle，哪项说法正确？

A) 只要上下文窗口足够大，中间信息就和开头信息一样可靠  
B) 关键证据应优先放在开头或结尾，并配合标题、摘要和结构化字段  
C) 工具输出越完整，模型越容易做出正确业务判断  
D) lost in the middle 只能通过换更大的模型解决

> **答案：B**  
> **解析：** 长上下文仍存在注意力位置偏差。工程上应通过关键位置、摘要、裁剪和结构化上下文降低风险。


## D5.2 — 设计升级模式与置信度校准

### 📖 概念解释

升级（escalation）和置信度校准是可靠 agent 的控制面。高风险业务不能只依赖模型“自称很有把握”。应把升级触发器写成明确规则：用户明确要求人工、政策或权限缺口、身份/合规风险、多轮无进展、候选对象无法消歧、工具不可用且无法恢复等。

CCA-F 常考区分：**情绪强烈不等于任务复杂**，情绪平静也可能是高风险任务。sentiment 可以作为服务优先级信号，但不应单独决定是否升级。置信度阈值应来自标注集校准：例如模型声称 high confidence 时，历史正确率是否足以自动执行。未达到阈值时，应 graceful degradation：承认限制、给出可执行下一步、收集缺失信息或交给人工。


In [ ]:
# Task 5.2：升级模式、置信度校准、HitL 阈值与 graceful degradation
# 注意：情绪分数不是复杂度，也不能单独作为人工升级依据。

calibration_rows = [
    {"case": "simple_refund", "self_confidence": 0.93, "correct": True},
    {"case": "policy_exception", "self_confidence": 0.91, "correct": False},
    {"case": "address_change", "self_confidence": 0.82, "correct": True},
    {"case": "ambiguous_identity", "self_confidence": 0.88, "correct": False},
]

def calibrated_precision_at(threshold: float) -> float | None:
    selected = [row for row in calibration_rows if row["self_confidence"] >= threshold]
    if not selected:
        return None
    return sum(row["correct"] for row in selected) / len(selected)

def decide_escalation(
    *,
    user_text: str,
    sentiment: str,
    task_complexity: str,
    policy_gap: bool,
    candidate_matches: int,
    attempts_without_progress: int,
    model_confidence: float,
) -> dict:
    reasons = []
    if re.search(r"人工|真人|客服|human|agent", user_text, re.I):
        reasons.append("explicit_human_request")
    if policy_gap:
        reasons.append("policy_gap")
    if candidate_matches > 1:
        reasons.append("ambiguous_multiple_matches")
    if attempts_without_progress >= 2:
        reasons.append("no_progress_after_retries")
    if task_complexity == "high" and model_confidence < 0.90:
        reasons.append("below_hitl_threshold_for_high_complexity")

    # 情绪只影响优先级，不单独决定升级。
    priority = "urgent" if sentiment == "angry" and reasons else "normal"
    action = "human_handoff" if reasons else "autonomous_resolution"
    fallback = "说明限制，保留已知事实，收集缺失字段或转人工。" if reasons else "继续自动处理。"
    return {"action": action, "priority": priority, "reasons": reasons, "fallback": fallback}

print("precision_at_0.90:", calibrated_precision_at(0.90))
print(json.dumps(decide_escalation(
    user_text="我不生气，但这个账户匹配到两个客户编号",
    sentiment="calm",
    task_complexity="high",
    policy_gap=False,
    candidate_matches=2,
    attempts_without_progress=0,
    model_confidence=0.87,
), ensure_ascii=False, indent=2))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 显式人工请求、政策缺口、权限不足、多次无进展、无法消歧时升级。
- HitL 阈值基于任务风险和校准集表现，而不是模型自评文本。
- sentiment 可用于优先级或话术调整，但不能替代复杂度和风险判断。
- graceful degradation 应说明限制、保存状态、提供下一步或结构化 handoff。

**✗ 常见陷阱**

- “用户很生气”就升级，“用户很平静”就不升级。
- 把模型输出的 `confidence: 0.95` 当作真实正确率。
- 用户明确要求人工时仍继续自动追问。
- 工具失败后只说“稍后再试”，不保留上下文和恢复路径。


### 🧪 模拟题

**Q1.** 用户语气平静，但同一个邮箱匹配到两个客户账户，且自动流程无法确定目标账户。最佳处理是什么？

A) 升级或触发 HitL，因为存在无法消歧的高风险匹配  
B) 不升级，因为用户情绪平静  
C) 随机选择最近登录的账户  
D) 让模型根据直觉选择置信度更高的账户

> **答案：A**  
> **解析：** 复杂度和风险来自身份消歧失败，而不是 sentiment。多重匹配无法可靠消歧时应升级或进入人工确认。

**Q2.** 关于模型置信度，下列哪项最符合 CCA-F？

A) 模型自报 high confidence 就可以自动执行高风险操作  
B) 置信度阈值应通过标注数据校准，并随任务风险设置 HitL 门槛  
C) sentiment 分数比校准数据更适合决定升级  
D) 只要模型解释充分，就不需要人工升级路径

> **答案：B**  
> **解析：** 自报置信度不是可靠概率。生产系统需要用验证集校准，并根据风险设定人工介入阈值。


## D5.3 — 实现跨 Agent 错误传播与信息来源链

### 📖 概念解释

跨 agent 协作的关键不是“每个子 agent 都成功”，而是失败、部分结果和来源能被上游正确理解。子 agent 失败时，应返回 `failure_type`、`attempted`、`partial_results`、`retryable`、`alternatives`；成功时，应为每个 claim 保留 source、摘录、时间或页码等 provenance。

协调者需要区分：哪些结论有来源支持，哪些只是推断，哪些区域存在 coverage gap。handoff 必须完整，包括当前目标、已尝试动作、失败原因、部分证据、下一步建议和风险。否则常见事故是：失败被空列表吞掉、来源在摘要中丢失、下游把未经证实的推断当事实。


In [ ]:
# Task 5.3：跨 agent 错误传播、provenance、attribution 与 handoff 完整性
# 目标：失败不被吞掉；成功 claim 必须能追溯到来源。

subagent_outputs = [
    {
        "agent": "web_search",
        "status": "failed",
        "failure_type": "timeout",
        "attempted": ["query: AI music market 2026", "query: enterprise adoption"],
        "partial_results": [{"title": "Market briefing", "url": "https://example.test/briefing"}],
        "retryable": True,
        "alternatives": ["retry with narrower query", "ask analyst to upload source report"],
    },
    {
        "agent": "doc_analysis",
        "status": "success",
        "claims": [
            {
                "claim": "The policy allows refund within 30 days.",
                "source": "refund_policy.pdf#page=4",
                "excerpt": "Refunds are permitted within 30 days of delivery.",
                "attribution": "doc_analysis",
            }
        ],
    },
]

def build_handoff(outputs: list[dict]) -> dict:
    claims = []
    coverage_gaps = []
    next_actions = []

    for item in outputs:
        if item["status"] == "success":
            for claim in item.get("claims", []):
                required = {"claim", "source", "attribution"}
                missing = sorted(required - set(claim))
                claims.append({**claim, "provenance_complete": not missing, "missing": missing})
        else:
            coverage_gaps.append({
                "agent": item["agent"],
                "failure_type": item["failure_type"],
                "attempted": item.get("attempted", []),
                "partial_results": item.get("partial_results", []),
                "retryable": item.get("retryable", False),
            })
            next_actions.extend(item.get("alternatives", []))

    return {"claims": claims, "coverage_gaps": coverage_gaps, "recommended_next_actions": next_actions}

print(json.dumps(build_handoff(subagent_outputs), ensure_ascii=False, indent=2))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- 子 agent 失败时结构化返回失败类型、已尝试动作、部分结果、是否可重试和替代方案。
- 每个 claim 保留来源、摘录/页码、归属 agent 和时间等 provenance。
- 协调者明确标注 coverage gaps，不把缺失证据包装成确定结论。
- handoff 包含目标、状态、证据、失败原因、下一步和风险。

**✗ 常见陷阱**

- 用空数组表示失败，导致上游误以为“没有发现问题”。
- 汇总时删除 source，使结论无法审计。
- 单个子任务失败就丢弃所有部分结果。
- 把 agent 的推断和来源事实混在一起。


### 🧪 模拟题

**Q1.** 搜索子 agent 超时，但已经找到一个可能相关的报告标题。它应该如何返回？

A) 返回 `failure_type`、`attempted`、`partial_results`、`retryable` 和 `alternatives`  
B) 返回空列表并标记成功，避免打断协调者  
C) 只返回“搜索失败”，不暴露细节  
D) 终止整个多 agent 工作流

> **答案：A**  
> **解析：** 结构化失败和部分结果能让协调者决定重试、替代检索、继续综合或升级。

**Q2.** 多 agent 汇总中，哪项最能支持 provenance 和 attribution？

A) 每个结论都携带 source、摘录/页码和产生该结论的 agent  
B) 最终摘要只保留结论，删除所有来源以节省 token  
C) 只记录最后一个 agent 的名字  
D) 只要协调者输出流畅，就不需要来源链

> **答案：A**  
> **解析：** provenance 让结论可追溯、可审计，也能帮助下游区分事实、推断和证据缺口。


## D5.4 — 使用 Batch API 与 Prompt Caching 优化成本和延迟

### 📖 概念解释

Batch API 和 prompt caching 都是成本/延迟优化手段，但适用边界不同。Batch 适合高吞吐、非阻塞、可等待的离线任务，例如夜间批量抽取、评测、回填。它不适合用户正在等待的阻塞流程，因为结果不是即时返回。

Prompt caching 适合重复长前缀：稳定 policy、schema、大段系统说明、工具规范或长文档背景。缓存 volatile 内容收益低，还可能增加复杂度。考试常考 tradeoff：Batch 可能降低单位成本但增加完成延迟；prompt caching 降低重复前缀成本和延迟，但只有当长前缀稳定且复用频繁时才值得。实时请求优先同步 API；离线批处理才考虑 Batch。


In [ ]:
# Task 5.4：Batch API、prompt caching、阻塞/非阻塞与成本延迟权衡
# 价格和折扣用示例数值表达相对关系；真实系统应以供应商当前价格表为准。

from math import ceil

def choose_execution_plan(*, blocking: bool, requests: int, repeated_prefix_tokens: int, deadline_hours: float) -> dict:
    batch_candidate = (not blocking) and requests >= 1000 and deadline_hours >= 1
    cache_candidate = repeated_prefix_tokens >= 1024
    if batch_candidate:
        mode = "message_batches_api"
        latency_profile = "non_blocking_completion_window"
    else:
        mode = "synchronous_messages_api"
        latency_profile = "interactive_or_low_volume"
    return {"mode": mode, "use_prompt_cache": cache_candidate, "latency_profile": latency_profile}

def estimate_relative_cost(*, requests: int, input_tokens: int, repeated_prefix_tokens: int, use_batch: bool, use_cache: bool) -> dict:
    # 教学用相对成本：batch 给总成本折扣，cache 给重复前缀折扣。
    batch_multiplier = 0.50 if use_batch else 1.00
    cached_prefix_multiplier = 0.10 if use_cache else 1.00
    variable_tokens = max(input_tokens - repeated_prefix_tokens, 0)
    effective_input_tokens = repeated_prefix_tokens * cached_prefix_multiplier + variable_tokens
    relative_cost_units = requests * effective_input_tokens * batch_multiplier
    return {
        "effective_input_tokens_per_request": ceil(effective_input_tokens),
        "relative_cost_units": ceil(relative_cost_units),
        "batch_multiplier": batch_multiplier,
        "cached_prefix_multiplier": cached_prefix_multiplier,
    }

plan = choose_execution_plan(blocking=False, requests=50_000, repeated_prefix_tokens=8_000, deadline_hours=12)
cost = estimate_relative_cost(requests=50_000, input_tokens=10_000, repeated_prefix_tokens=8_000, use_batch=True, use_cache=True)

message_shape = {
    "model": MODEL,
    "custom_id": "doc-0001",
    "messages": [{"role": "user", "content": "Extract invoice fields from document 0001."}],
    "system": [{"type": "text", "text": "Long stable extraction schema and policy...", "cache_control": {"type": "ephemeral"}}],
}

print(json.dumps({"plan": plan, "cost": cost, "message_shape": message_shape}, ensure_ascii=False, indent=2))


### ⚠️ 考点总结 & 易错点

**✅ 正确做法**

- Batch 用于高吞吐、非阻塞、可等待的离线任务，并用 `custom_id` 追踪结果。
- 阻塞式用户请求使用同步 Messages API 或流式响应，而不是 Batch。
- Prompt caching 用于稳定、重复、较长的前缀，例如 policy、schema、工具说明。
- 同时评估成本、延迟、重试、监控和失败恢复，而不是只看单次价格。

**✗ 常见陷阱**

- 为省钱把实时聊天、支付确认、客服等待流程改成 Batch。
- 缓存频繁变化的用户输入或短 prompt，收益很低。
- 忽略 Batch 的非即时完成特性和结果关联问题。
- 认为 prompt caching 会改善所有请求；它主要帮助重复长前缀。


### 🧪 模拟题

**Q1.** 每晚处理 5 万份合同，第二天上午交付结果，且每个请求共享同一份长 schema。最佳方案是什么？

A) 使用 Message Batches API，并对稳定 schema 使用 prompt caching  
B) 让用户界面同步等待所有合同处理完成  
C) 禁用缓存，避免复用任何前缀  
D) 把每份合同拆成多轮对话以提高可靠性

> **答案：A**  
> **解析：** 该任务高吞吐、非阻塞、可等待，适合 Batch；共享长 schema 稳定重复，适合 prompt caching。

**Q2.** 哪个场景最不适合 Batch API？

A) 用户正在结账页面等待支付风险判断  
B) 夜间批量生成 10 万条离线摘要  
C) 周末回填历史文档标签  
D) 离线运行大规模评测集

> **答案：A**  
> **解析：** 结账风险判断是阻塞式实时流程，不能依赖非即时完成的 Batch。Batch 更适合离线、延迟容忍任务。
